# Relationship Analysis & Time-Series Visualization

## Global COVID-19 Pandemic Analysis

## Introduction

This notebook explores relationships between key COVID-19 metrics and global trends using processed datasets. We'll investigate:

- **Relationship Analysis**: How GDP per capita, population density, and economic factors correlate with COVID-19 cases, deaths, and vaccination rates
- **Time-Series Trends**: Global patterns in new cases, deaths, and vaccination efforts over time
- **Continental Comparison**: How different continents experienced pandemic dynamics

All visualizations are created with Plotly for interactive exploration of the data.

## 1. Imports and Setup

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import nbformat
import warnings

warnings.filterwarnings("ignore")

# Configure pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

## 2. Define File Paths

In [2]:
# Define file paths
GLOBAL_SUMMARY_PATH = r"F:\faculty\Level 3 S_2\Data Visualization\Global COVID-19 Pandemic Analysis\Global-COVID-19-Pandemic-Analysis\Data\Processed\global_summary.csv"
FINAL_DATASET_PATH = r"F:\faculty\Level 3 S_2\Data Visualization\Global COVID-19 Pandemic Analysis\Global-COVID-19-Pandemic-Analysis\Data\Processed\final_dataset.csv"

# Fallback paths
RELATIVE_GLOBAL_SUMMARY = Path("../Data/Processed/global_summary.csv")
RELATIVE_FINAL_DATASET = Path("../Data/Processed/final_dataset.csv")

# Helper function to load data with fallback
def load_data(abs_path, rel_path):
    """Try absolute path first, then fallback to relative path"""
    abs_path_obj = Path(abs_path)
    if abs_path_obj.exists():
        return pd.read_csv(abs_path)
    elif rel_path.exists():
        return pd.read_csv(rel_path)
    else:
        raise FileNotFoundError(f"Could not find file using either {abs_path} or {rel_path}")

# Load datasets
global_summary_df = load_data(GLOBAL_SUMMARY_PATH, RELATIVE_GLOBAL_SUMMARY)
df = load_data(FINAL_DATASET_PATH, RELATIVE_FINAL_DATASET)

print("✓ Datasets loaded successfully")

✓ Datasets loaded successfully


## 3. Load and Explore Data

In [3]:
# Convert date column to datetime
df['date'] = pd.to_datetime(df['date'])

# Basic exploration
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"\nFinal Dataset Shape: {df.shape}")
print(f"Global Summary Shape: {global_summary_df.shape}")

print(f"\nDate Range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Unique Locations: {df['location'].nunique()}")
print(f"Unique Continents: {df['continent'].nunique()}")

# Check missing values for key visualization columns
key_cols = ['date', 'continent', 'location', 'population', 'gdp_per_capita', 
            'total_cases', 'total_deaths', 'cases_per_million', 'deaths_per_million', 
            'vaccination_rate_pct', 'people_fully_vaccinated_per_hundred', 
            'new_cases', 'new_vaccinations', 'cases_7day_ma', 'deaths_7day_ma']

print("\nMissing Values in Key Columns:")
missing = df[key_cols].isnull().sum()
for col in key_cols:
    if col in df.columns:
        missing_pct = (df[col].isnull().sum() / len(df)) * 100
        print(f"  {col}: {df[col].isnull().sum()} ({missing_pct:.1f}%)")
    else:
        print(f"  {col}: COLUMN NOT FOUND")

DATASET OVERVIEW

Final Dataset Shape: (395311, 76)
Global Summary Shape: (1688, 9)

Date Range: 2020-01-01 to 2024-08-14
Unique Locations: 237
Unique Continents: 6

Missing Values in Key Columns:
  date: 0 (0.0%)
  continent: 0 (0.0%)
  location: 0 (0.0%)
  population: 0 (0.0%)
  gdp_per_capita: 0 (0.0%)
  total_cases: 0 (0.0%)
  total_deaths: 0 (0.0%)
  cases_per_million: 0 (0.0%)
  deaths_per_million: 0 (0.0%)
  vaccination_rate_pct: 0 (0.0%)
  people_fully_vaccinated_per_hundred: 0 (0.0%)
  new_cases: 0 (0.0%)
  new_vaccinations: 0 (0.0%)
  cases_7day_ma: 0 (0.0%)
  deaths_7day_ma: 0 (0.0%)


## 4. Reusable Chart Styling Function

In [4]:
# Continent color mapping for relationship charts
continent_colors = {
    "Asia": "lightblue",
    "Europe": "lightcoral",
    "Africa": "lightgreen",
    "Oceania": "plum",
    "North America": "lightsalmon",
    "South America": "lightseagreen"
}

def apply_relationship_chart_guidelines(fig, title, x_title, y_title, y_zero=True):
    """
    Apply Week 5 lecture guidelines for relationship charts (Scatter, Bubble).
    
    Parameters:
    - fig: Plotly figure object
    - title: Chart title
    - x_title: X-axis label
    - y_title: Y-axis label
    - y_zero: If True, y-axis starts at 0 (for linear axes)
    """
    fig.update_layout(
        title=dict(
            text=title,
            x=0.5,
            xanchor="center",
            font=dict(size=22)
        ),
        template="plotly_white",
        plot_bgcolor="white",
        paper_bgcolor="white",
        font=dict(size=14),
        margin=dict(l=80, r=50, t=90, b=80),
        legend=dict(
            title="Continent",
            orientation="v",
            x=0.98,
            y=0.98,
            xanchor="right",
            yanchor="top",
            bgcolor="rgba(255,255,255,0.9)",
            bordercolor="black",
            borderwidth=1
        ),
        shapes=[
            dict(
                type="rect",
                xref="paper",
                yref="paper",
                x0=0,
                y0=0,
                x1=1,
                y1=1,
                line=dict(color="black", width=1.5),
                fillcolor="rgba(0,0,0,0)"
            )
        ]
    )

    fig.update_xaxes(
        title=x_title,
        showgrid=True,
        gridcolor="lightgray",
        zeroline=True,
        zerolinecolor="black",
        ticks="outside",
        tickangle=0
    )

    fig.update_yaxes(
        title=y_title,
        showgrid=True,
        gridcolor="lightgray",
        zeroline=True,
        zerolinecolor="black",
        ticks="outside",
        tickangle=0
    )

    if y_zero:
        fig.update_yaxes(rangemode="tozero")

    return fig

## 5. Prepare Data for Relationship Analysis

We'll create a dataset with the latest values for each country to analyze relationships between different metrics.

In [5]:
# Create dataset with latest values per country
latest_country_data = (df
    .dropna(subset=['continent', 'location'])
    .sort_values(['location', 'date'])
    .groupby('location')
    .tail(1)
    .reset_index(drop=True))

# Select needed columns for analysis
analysis_cols = ['location', 'continent', 'date', 'population', 'gdp_per_capita', 
                 'total_cases', 'total_deaths', 'cases_per_million', 'deaths_per_million', 
                 'vaccination_rate_pct', 'people_fully_vaccinated_per_hundred', 'population_density']

latest_country_data = latest_country_data[[col for col in analysis_cols if col in latest_country_data.columns]]

print(f"Latest country data shape: {latest_country_data.shape}")
print(f"Data ready for relationship analysis")

Latest country data shape: (237, 12)
Data ready for relationship analysis


In [6]:
# Create output directories for charts
output_dir = Path("../outputs/charts")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Output directory ready: {output_dir.resolve()}")

Output directory ready: F:\faculty\Level 3 S_2\Data Visualization\Global COVID-19 Pandemic Analysis\Global-COVID-19-Pandemic-Analysis\outputs\charts


## 6. Relationship Analysis: GDP vs Cases

This chart explores whether wealthier countries (higher GDP per capita) experienced different COVID-19 infection rates. We're using a logarithmic scale for GDP to better visualize the range of values. The size of each bubble represents the country's population.

In [7]:
# Prepare data for GDP vs Cases chart
gdp_cases_data = latest_country_data.dropna(subset=['gdp_per_capita', 'cases_per_million'])

fig_gdp_cases = px.scatter(
    gdp_cases_data,
    x='gdp_per_capita',
    y='cases_per_million',
    color='continent',
    color_discrete_map=continent_colors,
    hover_name='location',
    custom_data=['gdp_per_capita', 'cases_per_million', 'total_cases', 'vaccination_rate_pct', 'continent'],
    log_x=True,
    title='GDP per Capita vs COVID-19 Cases per Million',
    labels={'gdp_per_capita': 'GDP per Capita (USD, log scale)', 
            'cases_per_million': 'Cases per Million'},
    height=600
)

fig_gdp_cases.update_traces(
    marker=dict(
        size=7,
        opacity=0.75,
        line=dict(width=1, color='black')
    ),
    hovertemplate=(
        '<b>%{hovertext}</b><br>'
        '<b>Continent:</b> %{customdata[4]}<br>'
        '<b>GDP per Capita:</b> $%{customdata[0]:,.0f}<br>'
        '<b>Cases per Million:</b> %{customdata[1]:,.0f}<br>'
        '<b>Total Cases:</b> %{customdata[2]:,.0f}<br>'
        '<b>Vaccination Rate:</b> %{customdata[3]:.1f}%'
        '<extra></extra>'
    )
)

fig_gdp_cases = apply_relationship_chart_guidelines(fig_gdp_cases, 
    'GDP per Capita vs COVID-19 Cases per Million',
    'GDP per Capita (USD, log scale)',
    'Cases per Million',
    y_zero=True)

fig_gdp_cases.show()

# Save the chart
fig_gdp_cases.write_html(str(output_dir / "gdp_vs_cases.html"))
print("✓ Saved: outputs/charts/gdp_vs_cases.html")


✓ Saved: outputs/charts/gdp_vs_cases.html


## 7. Relationship Analysis: Population Size vs COVID-19 Deaths

This chart examines whether larger countries experienced proportionally more total deaths. Both axes use logarithmic scales to handle the wide range of values. Bubble size represents the death rate per million population.

In [8]:
# Prepare data for Population vs Deaths chart
pop_deaths_data = latest_country_data.dropna(subset=['population', 'total_deaths', 'deaths_per_million'])

fig_pop_deaths = px.scatter(
    pop_deaths_data,
    x='population',
    y='total_deaths',
    color='continent',
    color_discrete_map=continent_colors,
    hover_name='location',
    custom_data=['population', 'total_deaths', 'deaths_per_million', 'total_cases', 'continent'],
    log_x=True,
    title='Population Size vs Total COVID-19 Deaths',
    labels={'population': 'Population (log scale)', 
            'total_deaths': 'Total Deaths'},
    height=600
)

fig_pop_deaths.update_traces(
    marker=dict(
        size=7,
        opacity=0.75,
        line=dict(width=1, color='black')
    ),
    hovertemplate=(
        '<b>%{hovertext}</b><br>'
        '<b>Continent:</b> %{customdata[4]}<br>'
        '<b>Population:</b> %{customdata[0]:,.0f}<br>'
        '<b>Total Deaths:</b> %{customdata[1]:,.0f}<br>'
        '<b>Deaths per Million:</b> %{customdata[2]:.1f}<br>'
        '<b>Total Cases:</b> %{customdata[3]:,.0f}'
        '<extra></extra>'
    )
)

fig_pop_deaths = apply_relationship_chart_guidelines(fig_pop_deaths,
    'Population Size vs Total COVID-19 Deaths',
    'Population (log scale)',
    'Total Deaths',
    y_zero=True)

fig_pop_deaths.show()

# Save the chart
fig_pop_deaths.write_html(str(output_dir / "population_vs_deaths.html"))
print("✓ Saved: outputs/charts/population_vs_deaths.html")


✓ Saved: outputs/charts/population_vs_deaths.html


## 8. Relationship Analysis: GDP per Capita vs Vaccination Rate

Here we explore the relationship between economic development (GDP per capita) and vaccination progress. The chart shows whether wealthier nations were able to achieve higher vaccination rates. Bubble size represents population size.

In [9]:
# Prepare data for GDP vs Vaccination Rate chart
gdp_vax_data = latest_country_data.dropna(subset=['gdp_per_capita', 'vaccination_rate_pct', 'population'])

fig_gdp_vax = px.scatter(
    gdp_vax_data,
    x='gdp_per_capita',
    y='vaccination_rate_pct',
    color='continent',
    color_discrete_map=continent_colors,
    size='population',
    size_max=35,
    hover_name='location',
    custom_data=['gdp_per_capita', 'vaccination_rate_pct', 'population', 'people_fully_vaccinated_per_hundred', 'continent'],
    log_x=True,
    title='GDP per Capita vs Vaccination Rate',
    labels={'gdp_per_capita': 'GDP per Capita (USD, log scale)',
            'vaccination_rate_pct': 'Vaccination Rate (%)'},
    height=600
)

fig_gdp_vax.update_traces(
    marker=dict(
        sizemode="area",
        opacity=0.65,
        line=dict(width=1, color='black')
    ),
    hovertemplate=(
        '<b>%{hovertext}</b><br>'
        '<b>Continent:</b> %{customdata[4]}<br>'
        '<b>GDP per Capita:</b> $%{customdata[0]:,.0f}<br>'
        '<b>Vaccination Rate:</b> %{customdata[1]:.1f}%<br>'
        '<b>Population:</b> %{customdata[2]:,.0f}<br>'
        '<b>Fully Vaccinated per 100:</b> %{customdata[3]:.1f}'
        '<extra></extra>'
    )
)

fig_gdp_vax = apply_relationship_chart_guidelines(fig_gdp_vax,
    'GDP per Capita vs Vaccination Rate',
    'GDP per Capita (USD, log scale)',
    'Vaccination Rate (%)',
    y_zero=True)

fig_gdp_vax.show()

# Save the chart
fig_gdp_vax.write_html(str(output_dir / "gdp_vs_vaccination_rate.html"))
print("✓ Saved: outputs/charts/gdp_vs_vaccination_rate.html")


✓ Saved: outputs/charts/gdp_vs_vaccination_rate.html


## 9. Prepare Time-Series Data

Now we'll look at global trends over time by aggregating data across all countries for each date.

In [10]:
# Create global time-series data
global_time_data = (df
    .dropna(subset=['date'])
    .groupby('date')
    .agg({
        'new_cases': 'sum',
        'new_deaths': 'sum',
        'new_vaccinations': 'sum'
    })
    .reset_index())

# Calculate 7-day moving averages for smoothing
global_time_data['new_cases_7day_ma'] = global_time_data['new_cases'].rolling(window=7, min_periods=1).mean()
global_time_data['new_deaths_7day_ma'] = global_time_data['new_deaths'].rolling(window=7, min_periods=1).mean()
global_time_data['new_vaccinations_7day_ma'] = global_time_data['new_vaccinations'].rolling(window=7, min_periods=1).mean()

# Sort by date
global_time_data = global_time_data.sort_values('date').reset_index(drop=True)

print(f"Global time-series data shape: {global_time_data.shape}")
print(f"Date range: {global_time_data['date'].min().date()} to {global_time_data['date'].max().date()}")

Global time-series data shape: (1688, 7)
Date range: 2020-01-01 to 2024-08-14


## 10. Time-Series Analysis: Global New Cases Over Time

This chart shows the daily new COVID-19 cases reported worldwide. The smoothed line (7-day moving average) helps us see the overall trend more clearly by reducing day-to-day fluctuations.

In [11]:
fig_cases = go.Figure()

# Add raw cases data (lighter)
fig_cases.add_trace(go.Scatter(
    x=global_time_data['date'],
    y=global_time_data['new_cases'],
    name='Daily Cases',
    mode='lines',
    line=dict(color='rgba(99, 110, 250, 0.2)', width=1),
    hovertemplate='<b>Date:</b> %{x|%Y-%m-%d}<br><b>Daily Cases:</b> %{y:,.0f}<extra></extra>'
))

# Add 7-day moving average (bolder)
fig_cases.add_trace(go.Scatter(
    x=global_time_data['date'],
    y=global_time_data['new_cases_7day_ma'],
    name='7-Day Average',
    mode='lines',
    line=dict(color='rgb(99, 110, 250)', width=3),
    hovertemplate='<b>Date:</b> %{x|%Y-%m-%d}<br><b>7-Day Average:</b> %{y:,.0f}<extra></extra>'
))

fig_cases.update_layout(
    template="plotly_white",
    title={
        'text': 'Global New Cases Over Time',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18, 'color': '#333333'}
    },
    xaxis_title='Date',
    yaxis_title='New Cases',
    font=dict(size=12, family="Arial"),
    hovermode="x unified",
    margin=dict(l=80, r=50, t=100, b=80),
    height=500,
    xaxis=dict(showgrid=True, gridwidth=1, gridcolor='#f0f0f0'),
    yaxis=dict(showgrid=True, gridwidth=1, gridcolor='#f0f0f0')
)

fig_cases.show()

# Save the chart
fig_cases.write_html(str(output_dir / "global_new_cases_over_time.html"))
print("✓ Saved: outputs/charts/global_new_cases_over_time.html")

✓ Saved: outputs/charts/global_new_cases_over_time.html


## 11. Time-Series Analysis: Global Vaccination Trends

This chart displays the global daily vaccination efforts over time. The area chart helps visualize the volume and trends in vaccination campaigns worldwide.

In [12]:
fig_vax = px.area(
    global_time_data,
    x='date',
    y='new_vaccinations_7day_ma',
    title='Global Vaccination Trends Over Time',
    labels={'date': 'Date', 'new_vaccinations_7day_ma': 'New Vaccinations (7-day avg)'},
    height=500
)

fig_vax.update_traces(
    fillcolor='rgba(99, 110, 250, 0.3)',
    line=dict(color='rgb(99, 110, 250)', width=2),
    hovertemplate='<b>Date:</b> %{x|%Y-%m-%d}<br><b>Vaccinations:</b> %{y:,.0f}<extra></extra>'
)

fig_vax = style_plotly_chart(fig_vax,
    'Global Vaccination Trends Over Time',
    'Date',
    'New Vaccinations (7-day average)',
    hovermode='x unified')

fig_vax.show()

# Save the chart
fig_vax.write_html(str(output_dir / "global_vaccination_trends.html"))
print("✓ Saved: outputs/charts/global_vaccination_trends.html")

NameError: name 'style_plotly_chart' is not defined

## 12. Continental-Level Time-Series Analysis

Now let's compare how different continents experienced the pandemic in terms of cases and vaccinations.

In [ ]:
# Create continent-level time-series data
continent_time_data = (df
    .dropna(subset=['continent', 'date'])
    .groupby(['continent', 'date'])
    .agg({
        'new_cases': 'sum',
        'new_vaccinations': 'sum'
    })
    .reset_index())

# Calculate 7-day moving averages by continent
continent_time_data = continent_time_data.sort_values(['continent', 'date'])
continent_time_data['new_cases_7day_ma'] = (continent_time_data
    .groupby('continent')['new_cases']
    .rolling(window=7, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True))

continent_time_data['new_vaccinations_7day_ma'] = (continent_time_data
    .groupby('continent')['new_vaccinations']
    .rolling(window=7, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True))

continent_time_data = continent_time_data.reset_index(drop=True)

print(f"Continent time-series data shape: {continent_time_data.shape}")
print(f"Continents: {continent_time_data['continent'].unique().tolist()}")

Continent time-series data shape: (10080, 6)
Continents: ['Africa', 'Asia', 'Europe', 'North America', 'Oceania', 'South America']


### New Cases by Continent

In [ ]:
fig_cont_cases = px.line(
    continent_time_data,
    x='date',
    y='new_cases_7day_ma',
    color='continent',
    title='New Cases by Continent Over Time (7-Day Average)',
    labels={'date': 'Date', 'new_cases_7day_ma': 'New Cases', 'continent': 'Continent'},
    height=500
)

fig_cont_cases.update_traces(mode='lines', line=dict(width=2))

fig_cont_cases = style_plotly_chart(fig_cont_cases,
    'New Cases by Continent Over Time (7-Day Average)',
    'Date',
    'New Cases (7-day average)',
    hovermode='x unified')

fig_cont_cases.show()

# Save the chart
fig_cont_cases.write_html(str(output_dir / "continent_new_cases_over_time.html"))
print("✓ Saved: outputs/charts/continent_new_cases_over_time.html")

✓ Saved: outputs/charts/continent_new_cases_over_time.html


### Vaccination Trends by Continent

In [ ]:
fig_cont_vax = px.line(
    continent_time_data,
    x='date',
    y='new_vaccinations_7day_ma',
    color='continent',
    title='Vaccination Trends by Continent Over Time (7-Day Average)',
    labels={'date': 'Date', 'new_vaccinations_7day_ma': 'New Vaccinations', 'continent': 'Continent'},
    height=500
)

fig_cont_vax.update_traces(mode='lines', line=dict(width=2))

fig_cont_vax = style_plotly_chart(fig_cont_vax,
    'Vaccination Trends by Continent Over Time (7-Day Average)',
    'Date',
    'New Vaccinations (7-day average)',
    hovermode='x unified')

fig_cont_vax.show()

# Save the chart
fig_cont_vax.write_html(str(output_dir / "continent_vaccination_trends.html"))
print("✓ Saved: outputs/charts/continent_vaccination_trends.html")

✓ Saved: outputs/charts/continent_vaccination_trends.html


## Key Insights

### 1. Economic Development and Vaccination Rates
The relationship between GDP per capita and vaccination rates appears to show a positive correlation. Wealthier nations generally achieved higher vaccination coverage, which may suggest that economic resources played a significant role in vaccine procurement, distribution infrastructure, and vaccine hesitancy management.

### 2. Population Size and Absolute Deaths
The population vs. deaths chart suggests that larger nations, unsurprisingly, experienced more absolute deaths. However, when normalized per million population, the relationship becomes more complex. This indicates that absolute case counts depend heavily on population size, but death rates per capita may be influenced by other factors such as healthcare capacity and vaccination coverage.

### 3. GDP and Case Rates — No Clear Correlation
Interestingly, the GDP vs. cases chart does not show a strong linear relationship. Both wealthy and less wealthy nations experienced high case rates, suggesting that economic status alone did not guarantee lower infection rates. Other factors such as population density, early intervention strategies, and public health infrastructure may play equally important roles.

### 4. Continental Disparities in Case Waves
The continental case trends chart reveals distinct waves of infection across different continents, suggesting that pandemic dynamics varied significantly by region. Some continents experienced multiple waves while others showed more continuous transmission patterns, potentially reflecting differences in variant dominance, policy responses, and vaccination timing.

### 5. Vaccination Campaign Dynamics
Global vaccination trends show a marked increase from late 2020 through mid-2021, with peak vaccination efforts occurring several months after the pandemic's initial waves. The data suggests that vaccination campaigns ramped up considerably once vaccines became available, and this timing coincides with declining case growth rates in many regions.

### 6. Persistent Disparities Between Continents
Continent-level comparisons suggest that Africa and Asia experienced different pandemic trajectories compared to Europe and North America. These differences warrant further investigation into the role of vaccination access, healthcare infrastructure, population demographics, and public health policies in shaping pandemic outcomes.

## Conclusion

This analysis provides a comprehensive view of global COVID-19 pandemic patterns through relationship analysis and time-series visualization. The charts reveal complex interactions between economic development, population characteristics, vaccination efforts, and disease transmission.

Key findings highlight the importance of:
- **Vaccine Distribution**: Wealth correlation with vaccination rates suggests access inequity
- **Regional Variation**: Continental differences indicate that pandemic responses were highly contextual
- **Temporal Dynamics**: Multiple waves and vaccination campaigns show how the pandemic evolved over time

For future analysis, consider investigating:
- Case fatality rates stratified by vaccination status
- Healthcare capacity and its correlation with death rates
- Policy interventions and their timing relative to infection waves
- Socioeconomic factors beyond GDP that may influence pandemic outcomes

---

**All charts have been saved to the `outputs/charts/` directory as interactive HTML files.**